In [1]:
#py_data_analysis environment
import os
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from pyproj import Transformer
from bilinear_interpolation import bilinear_interpolation
import xarray as xr
import rioxarray as rio 
from rasterio.enums import Resampling

### Read data

In [2]:
src_path = r"W:/VUB/_main_research/data/RMI"
os.chdir(src_path)

# Read gridded meteo data with encoding ISO-8859-1 since utf-8 doesnt work (try other encodings e.g. latin1, cp1252)
try:
    clim_data = pd.read_csv("pdg1487.csv", sep=";", header=0, encoding="ISO-8859-1")
except UnicodeDecodeError as e:
    print("ISO-8859-1 didn't work: ", e)

#read 5km by 5km grid
grid = pd.read_csv(src_path+"/gridded_data_docu/grid 5x5km_def.csv", sep=" ", skiprows=1)

In [3]:
#check uniquee pairs of projected coordinates
y_unique = grid["LAMBERT_Y"].unique()
x_unique = grid["LAMBERT_X"].unique()

shape = (x_unique.shape, y_unique.shape)

#check if the lat/lon coordinates are unique
lat_unique = grid["LAT"].unique()
lon_unique = grid["LON"].unique()

shape_latlon = (lon_unique.shape,lat_unique.shape)

print(shape)
print(shape_latlon)

((56,), (45,))
((888,), (514,))


### Correct lat/lon coordinates to create regular grid
Lat,lon values for the same lat/lon position have small differences in the order of 0.000x so they will not form a uniform grid.  
We will group the lat, lon values according to LAMBERT_X and LAMBERT_Y

In [ ]:
for lat in grid['LAMBERT_Y']:
    #select lat for this row. These lat groups should be the same.
    
    df=grid[grid['LAMBERT_Y']==lat]['LAT']
    mean_lat = df.mean()
    
    #Replace each lat value with the mean of the group
    grid.loc[grid['LAMBERT_Y']==lat, 'LAT'] = mean_lat

for lon in grid['LAMBERT_X']:
    #select lon for this col. These lon groups should be the same.
     #Replace each lon value with the mean of the group
    df=grid[grid['LAMBERT_X']==lon]['LON']
    mean_lon = df.mean()
    #replace df values with mean_lon
    grid.loc[grid['LAMBERT_X']==lon, 'LON'] = mean_lon

#### Merge data and coordinates files

In [5]:
#merge the two dataframes on pixel_id
climate_data_df = clim_data.merge(grid, on='PIXEL_ID')

#sort the index
climate_data_df.sort_index(inplace=True)

#rename the columns to lower case
climate_data_df.columns = climate_data_df.columns.str.lower()

#### Rename columns

In [6]:
#extract climate variable names from the dataframe columns
climate_variables=['tmax', 'tmin', 
                   'prec', 'PET',
                   'global_radiation', 'wind_speed']

# Replace column names from index 2 to 8 (clim variables)
subset_columns = list(climate_data_df.columns[4:10])
new_names_dict = dict(zip(subset_columns, climate_variables))

# Rename the subset of columns
climate_data_df.rename(columns=new_names_dict, inplace=True)

In [7]:
climate_data_df[climate_data_df['lambert_x']==62500].head()

,date,pixel_id,pixel_lon_center,pixel_lat_center,tmax,tmin,prec,PET,global_radiation,wind_speed,lambert_x,lambert_y,lat,lon
0,1994-01-01,1,5.419101,49.510015,4.1,2.3,10.4,0.4,0.37,4.9,62500,6218014.18,49.509000,5.435683
4,1994-01-01,5,5.419910,49.554969,3.3,2.2,10.3,0.4,0.38,4.9,62500,6223014.18,49.553143,5.435683
13,1994-01-01,14,5.420720,49.599924,3.5,1.8,10.7,0.4,0.38,4.9,62500,6228014.18,49.598200,5.435683
23,1994-01-01,24,5.421531,49.644879,3.4,1.5,11.0,0.4,0.38,4.9,62500,6233014.18,49.643200,5.435683
35,1994-01-01,36,5.422344,49.689835,3.6,1.2,11.2,0.4,0.39,4.9,62500,6238014.18,49.688750,5.435683


#### Drop extra columns and assign index

In [8]:
#drop columns that are not needed
climate_data_df.drop(columns=['pixel_id','lambert_x', 'lambert_y','pixel_lon_center','pixel_lat_center'], inplace=True)

#rename date column to time
climate_data_df.rename(columns={'date':'time'}, inplace=True)

#convert the date column to datetime
climate_data_df['time'] = pd.to_datetime(climate_data_df['time'], format='%Y-%m-%d')

#set the time, lat and lon as index
climate_data_df.set_index(['time', 'lat', 'lon'], inplace=True)

#### Convert pd DataFrame to xarray Dataset

In [9]:
ds = xr.Dataset.from_dataframe(climate_data_df)

#calculate average temperature tavg
ds['tavg'] = (ds['tmax'] + ds['tmin']) / 2

In [10]:
#assign projection to the dataset
ds.rio.write_crs("EPSG:4326", inplace=True) #Activate this line if Option 1 is used
ds = ds.rio.set_spatial_dims(x_dim="lon", y_dim="lat")
ds = ds.rio.reproject(
    dst_crs="EPSG:4326",
)
#rename the coords
ds = ds.rename({'y': 'lat', 'x': 'lon'})

In [11]:
ds = ds.rename_vars({'prec': 'pre', 'PET': 'pet'})
#assign -9999 to fill values
ds = ds.fillna(-9999)
#set fill values
ds.attrs['_FillValue'] = -9999

#### Change time from pd.datetime to int32 format (CF conventions)

In [12]:
#change time to days since 1900-01-01
ds['time'] = (ds['time'] - np.datetime64('1900-01-01')).dt.total_seconds() / (3600 * 24)
ds['time'].attrs['units'] = 'days since 1900-01-01'

#### Assign attributes

In [13]:
ds.time.attrs['calendar'] = 'gregorian'
ds.time.attrs['standard_name'] = 'time'
ds.time.attrs['long_name'] = 'time'

ds['lat'].attrs['units'] = 'degrees_north'
ds['pre'].attrs['units'] = 'mm'
ds['pre'].attrs['long_name'] = 'Total Precipitation'
ds['pet'].attrs['units'] = 'mm'
ds['pet'].attrs['long_name'] = 'Potential Evapotranspiration'
ds['tavg'].attrs['units'] = 'degrees C'
ds['tavg'].attrs['long_name'] = 'Average Temperature'

ds['pet'].attrs['_FillValue'] = -9999.0
ds['pre'].attrs['_FillValue'] = -9999.0
ds['tavg'].attrs['_FillValue'] = -9999.0

In [14]:
print(ds['pet'].dims)  # Should return ('time', 'lat', 'lon')
print(ds['pre'].dims)  # Should return ('time', 'lat', 'lon')
print(ds['tavg'].dims)  # Should return ('time', 'lat', 'lon')

('time', 'lat', 'lon')
('time', 'lat', 'lon')
('time', 'lat', 'lon')


#### Export for mHM

In [15]:
### Export mHM netcdf files
vars = ['pre', 'pet','tavg']

#export the dataset to netcdf
for var in vars:
    ds[[var]].to_netcdf(f"W:/VUB/_main_research/mHM/mhm_belgium/RMI/mHM_intermediate_files/RMI_{var}_grid.nc")